# ClarioAI System Test Notebook

Notebook ini menguji seluruh komponen sistem ClarioAI secara terstruktur:

| # | Seksi | Deskripsi |
|---|-------|-----------|
| 1 | Imports & Environment | Verifikasi semua package tersedia |
| 2 | State Schema | Dataclass dan EBPState |
| 3 | Utility Functions | `extract_json`, `format_constraints` |
| 4 | LLM Connectivity | Koneksi ke DeepInfra / Qwen |
| 5 | Internet Search Tool | BrightData API |
| 6 | MongoDB | Koneksi, save, load state |
| 7 | Agent Nodes | Market Scout, Strategic Architect, Financial Analyst, Ethics Agent, Orchestrator |
| 8 | Graph Pipeline | Build graph & jalankan 1 iterasi penuh |
| 9 | End-to-End | Full run dengan business scenario nyata |

---
## 1. Imports & Environment Check

In [1]:
import sys, importlib

REQUIRED_PACKAGES = [
    "langchain",
    "langchain_openai",
    "langchain_core",
    "langgraph",
    "pymongo",
    "requests",
]

missing = []
for pkg in REQUIRED_PACKAGES:
    try:
        importlib.import_module(pkg)
        print(f"  [OK] {pkg}")
    except ImportError:
        print(f"  [MISSING] {pkg}")
        missing.append(pkg)

if missing:
    print(f"\nInstall missing: pip install {' '.join(missing)}")
else:
    print("\nSemua package tersedia.")

  [OK] langchain
  [OK] langchain_openai
  [OK] langchain_core
  [OK] langgraph
  [OK] pymongo
  [OK] requests

Semua package tersedia.


In [2]:
# Pastikan root project ada di sys.path agar import relatif berjalan
import os
PROJECT_ROOT = os.path.abspath(".")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

Project root: d:\Gabriel\Gabriel\Telkom University\Kuliah\Semester VI\Capstone\Multi-Agent-Decision-Support-System


---
## 2. State Schema Tests

In [3]:
from states.schema import (
    BussinessConstraints,
    MarketScoutReport,
    StrategicReport,
    FinancialAnalysisReport,
    EthicsAnalysisReport,
    EBPState,
)
print("Import states.schema: OK")

Import states.schema: OK


In [4]:
# --- Test BussinessConstraints (intentional typo sesuai codebase) ---
constraints = BussinessConstraints(
    sector_and_domain="EdTech / Platform Belajar Online",
    audience="Pelajar SMA dan mahasiswa di Indonesia (usia 16-22)",
    initial_prompt="Platform micro-learning berbasis video pendek untuk persiapan SNBT dan ujian sertifikasi profesional",
)
print("BussinessConstraints:")
print(f"  sector_and_domain : {constraints.sector_and_domain}")
print(f"  audience          : {constraints.audience}")
print(f"  initial_prompt    : {constraints.initial_prompt}")

BussinessConstraints:
  sector_and_domain : EdTech / Platform Belajar Online
  audience          : Pelajar SMA dan mahasiswa di Indonesia (usia 16-22)
  initial_prompt    : Platform micro-learning berbasis video pendek untuk persiapan SNBT dan ujian sertifikasi profesional


In [6]:
# --- Test report dataclasses ---
market_report = MarketScoutReport(
    ideas=["Ide A", "Ide B"],
    agent_explanation="Penjelasan dari market scout.",
    sources=["internal_brainstorm", "external_search"],
)
strategic_report = StrategicReport(
    swot_analysis="SWOT placeholder",
    pastel_analysis="PESTEL placeholder",
)
financial_report = FinancialAnalysisReport(analysis_result="Proyeksi keuangan placeholder")
ethics_report    = EthicsAnalysisReport(analysis_result="Analisis etika placeholder")

print("MarketScoutReport  :", market_report)
print("StrategicReport    :", strategic_report)
print("FinancialAnalysis  :", financial_report)
print("EthicsAnalysis     :", ethics_report)

MarketScoutReport  : MarketScoutReport(ideas=['Ide A', 'Ide B'], agent_explanation='Penjelasan dari market scout.', sources=['internal_brainstorm', 'external_search'])
StrategicReport    : StrategicReport(swot_analysis='SWOT placeholder', pastel_analysis='PESTEL placeholder')
FinancialAnalysis  : FinancialAnalysisReport(analysis_result='Proyeksi keuangan placeholder')
EthicsAnalysis     : EthicsAnalysisReport(analysis_result='Analisis etika placeholder')


In [7]:
# --- Test EBPState instantiation ---
import uuid

sample_state: EBPState = {
    "state_id": str(uuid.uuid4()),
    "user_id": "test_user_notebook",
    "bussiness_constraints": constraints,
    "market_scout_report": None,
    "strategic_report": None,
    "financial_analysis_report": None,
    "ethics_analysis_report": None,
    "approval_status": "pending",
    "orchestrator_feedback": None,
    "messages": [],
    "iteration": 0,
    "max_iterations": 3,
    "user_feedback": None,
}
print("EBPState state_id :", sample_state["state_id"])
print("approval_status   :", sample_state["approval_status"])
print("iteration         :", sample_state["iteration"], "/", sample_state["max_iterations"])
print("market_report None:", sample_state["market_scout_report"] is None)

EBPState state_id : d8b9c4c1-0cde-4778-881d-c003555f26c7
approval_status   : pending
iteration         : 0 / 3
market_report None: True


---
## 3. Utility Functions Tests

In [8]:
from functions.agent_utils import extract_json, format_constraints
print("Import functions.agent_utils: OK")

Import functions.agent_utils: OK


In [9]:
# --- extract_json: markdown fenced ---
fenced = '''
Berikut hasilnya:
```json
{"status": "approved", "score": 8.5, "reason": "Market viable"}
```
'''
result = extract_json(fenced)
assert result["status"] == "approved", "Fenced JSON parse gagal"
assert result["score"] == 8.5
print("extract_json (fenced)  :", result)

extract_json (fenced)  : {'status': 'approved', 'score': 8.5, 'reason': 'Market viable'}


In [10]:
# --- extract_json: bare JSON ---
bare = 'Analisis selesai. {"ideas": ["A", "B"], "confidence": 0.9}'
result2 = extract_json(bare)
assert result2["ideas"] == ["A", "B"]
print("extract_json (bare)    :", result2)

extract_json (bare)    : {'ideas': ['A', 'B'], 'confidence': 0.9}


In [11]:
# --- extract_json: fallback ke {"raw": ...} ---
plain = "Tidak ada JSON di sini sama sekali."
result3 = extract_json(plain)
assert "raw" in result3
print("extract_json (fallback):", result3)

extract_json (fallback): {'raw': 'Tidak ada JSON di sini sama sekali.'}


In [12]:
# --- format_constraints ---
formatted = format_constraints(constraints)
print("format_constraints output:")
print(formatted)
assert "EdTech" in formatted
assert "Pelajar SMA" in formatted

format_constraints output:
Sector/Domain: EdTech / Platform Belajar Online
Target Audience: Pelajar SMA dan mahasiswa di Indonesia (usia 16-22)
Business Idea: Platform micro-learning berbasis video pendek untuk persiapan SNBT dan ujian sertifikasi profesional


In [13]:
# --- format_constraints dengan None ---
none_fmt = format_constraints(None)
assert "No constraints" in none_fmt
print("format_constraints(None):", none_fmt)

format_constraints(None): No constraints provided.


---
## 4. LLM Connectivity Test

> Sel ini melakukan satu panggilan ke DeepInfra API. Perlu koneksi internet.

In [14]:
from functions.llm import get_llm, MODEL_NAME, DEEPINFRA_BASE_URL
print(f"Model      : {MODEL_NAME}")
print(f"Base URL   : {DEEPINFRA_BASE_URL}")

Model      : Qwen/Qwen3.5-397B-A17B
Base URL   : https://api.deepinfra.com/v1/openai


In [15]:
llm = get_llm(temperature=0.3)

from langchain_core.messages import HumanMessage

response = llm.invoke([HumanMessage(content="Balas hanya dengan kata: PING")])
print("LLM response:", response.content[:200])

LLM response: 

PING


---
## 5. Internet Search Tool Test

> Memerlukan koneksi internet dan BrightData API key yang valid.

In [16]:
from tools.internet_search import internet_search
print("Import tools.internet_search: OK")
print("Tool name :", internet_search.name)
print("Tool desc :", internet_search.description[:80], "...")

Import tools.internet_search: OK
Tool name : internet_search
Tool desc : Search the internet for real-time business and market information.
    Use this  ...


In [17]:
query = "Ide bisnis kuliner"
print(f"Searching: '{query}' ...\n")

search_result = internet_search.invoke({"query": query})
# print(search_result[:1000])  # Tampilkan 1000 char pertama

[12:19:53] [internet_search] START query='Ide bisnis kuliner'
[12:19:53] Starting new HTTPS connection (1): api.brightdata.com:443


Searching: 'Ide bisnis kuliner' ...



[12:19:59] https://api.brightdata.com:443 "POST /request HTTP/1.1" 200 None
[12:19:59] [internet_search] BrightData search took 5.77s, status=200
[12:19:59] [internet_search] Parsed 5 results
[12:19:59] Starting new HTTPS connection (1): www.unileverfoodsolutions.co.id:443
[12:19:59] Starting new HTTPS connection (1): www.lalamove.com:443
[12:19:59] Starting new HTTPS connection (1): ocean.bca.co.id:443
[12:19:59] Starting new HTTPS connection (1): www.foliopos.com:443
[12:19:59] Starting new HTTPS connection (1): www.cimbniaga.co.id:443
[12:20:00] https://www.unileverfoodsolutions.co.id:443 "GET /id/inspirasi-chef/ukm-sukses/10-ide-jualan-makanan-kekinian-untuk-bisnis-kuliner.html HTTP/1.1" 403 524
[12:20:00] [_fetch_page_text] HTTP GET took 1.16s, status=403, url=https://www.unileverfoodsolutions.co.id/id/inspirasi-chef/ukm-sukses/10-ide-jualan-makanan-kekinian-untuk-bisnis-kuliner.html
[12:20:00] https://ocean.bca.co.id:443 "GET /artikel/10-ide-bisnis-kuliner HTTP/1.1" 307 None
[12:

In [18]:
print(search_result)

Search results for 'Ide bisnis kuliner':


1. 30 Ide Jualan Makanan Kekinian Untuk Bisnis Kuliner!
   https://www.unileverfoodsolutions.co.id/id/inspirasi-chef/ukm-sukses/10-ide-jualan-makanan-kekinian-untuk-bisnis-kuliner.html

2. 10 Ide Bisnis Kuliner Kekinian, Bisa Dicoba untuk Pemula
   https://ocean.bca.co.id/artikel/10-ide-bisnis-kuliner

3. 25 Ide Usaha Kuliner Rumahan yang Mudah dan ...
   https://www.foliopos.com/blog/detail/25-ide-usaha-kuliner-rumahan-yang-mudah-dan-menguntungkan

4. 23 Ide Bisnis Kuliner Modal Kecil Tapi Cuan Besar!
   https://www.lalamove.com/id/blog/bisnis-kuliner-modal-kecil/

5. Ide Usaha Makanan Kekinian dan Strategi Bisnisnya
   https://www.cimbniaga.co.id/id/inspirasi/bisnis/ide-usaha-makanan-kekinian-dan-strategi-bisnisnya


-- Page Summaries --

[https://ocean.bca.co.id/artikel/10-ide-bisnis-kuliner]
Here’s a concise summary of the webpage content relevant to the search query “Ide bisnis kuliner”:

**The webpage provides 10 ideas for starting a suc

In [19]:
stop

NameError: name 'stop' is not defined

---
## 6. MongoDB Tests

> Memerlukan MongoDB berjalan di `localhost:27017`.

In [20]:
from functions.mongodb import create_new_state, save_state, load_state, MONGO_URI, DB_NAME
print(f"MongoDB URI : {MONGO_URI}")
print(f"Database    : {DB_NAME}")

MongoDB URI : mongodb://localhost:27017
Database    : clario_ai


In [21]:
# --- Test koneksi MongoDB ---
from pymongo import MongoClient
try:
    client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=3000)
    client.admin.command("ping")
    print("MongoDB connection: OK")
    MONGO_AVAILABLE = True
except Exception as e:
    print(f"MongoDB connection: GAGAL ({e})")
    print("Test seksi ini di-skip. Jalankan MongoDB terlebih dahulu.")
    MONGO_AVAILABLE = False

MongoDB connection: OK


In [22]:
if MONGO_AVAILABLE:
    # --- create_new_state ---
    new_state = create_new_state(
        constraints=constraints,
        user_id="notebook_test_user",
        max_iterations=3,
    )
    print("State dibuat, ID:", new_state["state_id"])
    print("approval_status :", new_state["approval_status"])
    print("iteration       :", new_state["iteration"])
    TEST_STATE_ID = new_state["state_id"]
else:
    TEST_STATE_ID = None
    print("Skip — MongoDB tidak tersedia.")

State dibuat, ID: b2c13602-95cd-43fc-b505-426e92295bb4
approval_status : pending
iteration       : 0


In [23]:
if MONGO_AVAILABLE:
    # --- save_state ---
    new_state["market_scout_report"] = market_report
    saved_id = save_state(new_state)
    print("save_state OK, returned ID:", saved_id)
else:
    print("Skip.")

save_state OK, returned ID: b2c13602-95cd-43fc-b505-426e92295bb4


In [24]:
if MONGO_AVAILABLE:
    # --- load_state ---
    loaded = load_state(TEST_STATE_ID)
    assert loaded is not None, "State tidak ditemukan setelah save!"
    assert loaded["state_id"] == TEST_STATE_ID
    assert loaded["market_scout_report"] is not None
    assert loaded["market_scout_report"].ideas == ["Ide A", "Ide B"]
    print("load_state OK")
    print("  state_id        :", loaded["state_id"])
    print("  user_id         :", loaded["user_id"])
    print("  market ideas    :", loaded["market_scout_report"].ideas)

    # --- load non-existent ---
    not_found = load_state("00000000-0000-0000-0000-000000000000")
    assert not_found is None
    print("  load non-existent: None (benar)")
else:
    print("Skip.")

load_state OK
  state_id        : b2c13602-95cd-43fc-b505-426e92295bb4
  user_id         : notebook_test_user
  market ideas    : ['Ide A', 'Ide B']
  load non-existent: None (benar)


In [25]:
if MONGO_AVAILABLE:
    # --- Serialisasi messages (HumanMessage, AIMessage, SystemMessage) ---
    from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
    from functions.mongodb import _serialize_messages, _deserialize_messages

    msgs = [
        SystemMessage(content="System prompt"),
        HumanMessage(content="Halo"),
        AIMessage(content="Halo juga!"),
    ]
    serialized = _serialize_messages(msgs)
    deserialized = _deserialize_messages(serialized)

    assert len(deserialized) == 3
    assert deserialized[1].content == "Halo"
    print("Message serialization/deserialization: OK")
    print("  Types:", [type(m).__name__ for m in deserialized])
else:
    print("Skip.")

Message serialization/deserialization: OK
  Types: ['SystemMessage', 'HumanMessage', 'AIMessage']


---
## 7. Agent Node Tests

Setiap agent diuji secara **terpisah** dengan state minimal. Ini menguji logika prompt + parsing output tiap node tanpa menjalankan graph penuh.

> Sel-sel ini memanggil LLM secara nyata — estimasi 30-120 detik per agent.

In [26]:
# Helper: buat state bersih untuk setiap test
def make_fresh_state(iteration=0, with_market=False, with_strategic=False, with_financial=False, with_ethics=False):
    state: EBPState = {
        "state_id": str(uuid.uuid4()),
        "user_id": "notebook_agent_test",
        "bussiness_constraints": constraints,
        "market_scout_report": market_report if with_market else None,
        "strategic_report": strategic_report if with_strategic else None,
        "financial_analysis_report": financial_report if with_financial else None,
        "ethics_analysis_report": ethics_report if with_ethics else None,
        "approval_status": "pending",
        "orchestrator_feedback": None,
        "messages": [],
        "iteration": iteration,
        "max_iterations": 3,
        "user_feedback": None,
    }
    return state

print("Helper make_fresh_state: siap")

Helper make_fresh_state: siap


### 7.1 Market Scout Agent

In [27]:
from nodes.market_scout import market_scout_node

state_ms = make_fresh_state(iteration=0)
print("Menjalankan Market Scout Agent...")
result_ms = market_scout_node(state_ms)

report_ms = result_ms.get("market_scout_report")
assert report_ms is not None, "market_scout_report seharusnya tidak None!"
assert isinstance(report_ms.ideas, list), "ideas harus berupa list"
assert len(report_ms.ideas) > 0, "ideas tidak boleh kosong"

print("\n[Market Scout] LULUS")
print(f"  Jumlah ide       : {len(report_ms.ideas)}")
for i, idea in enumerate(report_ms.ideas, 1):
    print(f"  Ide {i}: {idea[:100]}")
print(f"\n  Penjelasan (150 char): {report_ms.agent_explanation[:150]}...")

2026-05-27 12:21:49.870  DEBUG  [clario.market_scout_enterprise]  ============================================================
2026-05-27 12:21:49.871  DEBUG  [clario.market_scout_enterprise]  -> Enterprise Market Scout Agent Dimulai
2026-05-27 12:21:49.885  DEBUG  [clario.enterprise_market_scout]  [PLAN] Merencanakan 4 query pencarian...
[12:21:49] Request options: {'method': 'post', 'url': '/chat/completions', 'headers': {'X-Stainless-Raw-Response': 'true'}, 'files': None, 'idempotency_key': 'stainless-python-retry-82834b49-9f63-4605-8b98-bd1b9eb905b6', 'security': {'bearer_auth': True}, 'content': None, 'json_data': {'messages': [{'content': 'You are the Senior Market Scout Agent in an enterprise business planning system.\nYour mission is to rigorously validate whether a target market is viable to enter by transforming bulk search data into highly actionable market intelligence.\n\nYour analysis must be synthesized entirely in Bahasa Indonesia and formatted into a structured JSON re

Menjalankan Market Scout Agent...


[12:21:50] start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x000001FF1B842E10>
[12:21:50] send_request_headers.started request=<Request [b'POST']>
[12:21:50] send_request_headers.complete
[12:21:50] send_request_body.started request=<Request [b'POST']>
[12:21:50] send_request_body.complete
[12:21:50] receive_response_headers.started request=<Request [b'POST']>
[12:22:27] receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Wed, 27 May 2026 05:22:28 GMT'), (b'Content-Type', b'application/json'), (b'Content-Length', b'14800'), (b'Connection', b'keep-alive'), (b'server', b'uvicorn'), (b'x-robots-tag', b'noindex'), (b'x-request-id', b'RdyoYzuKHU3X0PaFMeqMrmxU')])
[12:22:27] HTTP Request: POST https://api.deepinfra.com/v1/openai/chat/completions "HTTP/1.1 200 OK"
[12:22:27] receive_response_body.started request=<Request [b'POST']>
[12:22:27] receive_response_body.complete
[12:22:27] response_closed.started
[12:22:27] response_cl


[Market Scout] LULUS
  Jumlah ide       : 2
  Ide 1: Platform micro-learning video pendek (3-5 menit) khusus persiapan SNBT dengan fitur adaptive learnin
  Ide 2: Layanan bundled micro-learning untuk sertifikasi profesional (TOEFL, AWS, Google Certifications) den

  Penjelasan (150 char): Berdasarkan analisis multi-source, pasar EdTech Indonesia untuk platform micro-learning persiapan SNBT dan sertifikasi profesional menunjukkan kelayak...


### 7.2 Strategic Architect Agent

In [28]:
from nodes.strategic_architect import strategic_architect_node

# Strategic Architect butuh market_scout_report
state_sa = make_fresh_state(iteration=0, with_market=True)
# Gunakan hasil nyata dari market scout jika tersedia
if report_ms is not None:
    state_sa["market_scout_report"] = report_ms

print("Menjalankan Strategic Architect Agent...")
result_sa = strategic_architect_node(state_sa)

report_sa = result_sa.get("strategic_report")
assert report_sa is not None, "strategic_report seharusnya tidak None!"
assert report_sa.swot_analysis, "swot_analysis tidak boleh kosong"
assert report_sa.pastel_analysis, "pastel_analysis tidak boleh kosong"

print("\n[Strategic Architect] LULUS")
print(f"  SWOT (200 char)  : {report_sa.swot_analysis[:200]}...")
print(f"  PESTEL (200 char): {report_sa.pastel_analysis[:200]}...")

2026-05-27 12:23:56.031  DEBUG  [clario.strategic_architect_enterprise]  ============================================================
2026-05-27 12:23:56.032  DEBUG  [clario.strategic_architect_enterprise]  -> Enterprise Strategic Architect Agent Dimulai
2026-05-27 12:23:56.040  DEBUG  [clario.enterprise_strategic_architect]  [PLAN] Merencanakan 3 query pencarian...
[12:23:56] Request options: {'method': 'post', 'url': '/chat/completions', 'headers': {'X-Stainless-Raw-Response': 'true'}, 'files': None, 'idempotency_key': 'stainless-python-retry-c028f13b-2c2c-4b5f-b733-12b8b9ff345f', 'security': {'bearer_auth': True}, 'content': None, 'json_data': {'messages': [{'content': 'You are the Senior Strategic Architect Agent in an enterprise business planning system.\nYour mission is to build data-backed, non-generic strategic execution models that prevent businesses from failing due to poor prioritization.\n\nYour analysis must be synthesized entirely in Bahasa Indonesia and returned in a str

Menjalankan Strategic Architect Agent...


[12:23:56] start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x000001FF1BDAFE00>
[12:23:56] send_request_headers.started request=<Request [b'POST']>
[12:23:56] send_request_headers.complete
[12:23:56] send_request_body.started request=<Request [b'POST']>
[12:23:56] send_request_body.complete
[12:23:56] receive_response_headers.started request=<Request [b'POST']>
[12:24:28] receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Wed, 27 May 2026 05:24:28 GMT'), (b'Content-Type', b'application/json'), (b'Content-Length', b'11364'), (b'Connection', b'keep-alive'), (b'server', b'uvicorn'), (b'x-robots-tag', b'noindex'), (b'x-request-id', b'RvMG7X49DV4Bl6Hwrt4cAB0B')])
[12:24:28] HTTP Request: POST https://api.deepinfra.com/v1/openai/chat/completions "HTTP/1.1 200 OK"
[12:24:28] receive_response_body.started request=<Request [b'POST']>
[12:24:28] receive_response_body.complete
[12:24:28] response_closed.started
[12:24:28] response_cl


[Strategic Architect] LULUS
  SWOT (200 char)  : ## Analisis SWOT Komprehensif

**Strengths:**
- Format micro-learning video 3-5 menit selaras dengan perilaku konsumsi konten Gen Z (rata-rata sesi TikTok 1 jam 53 menit/hari, YouTube 16 menit 49 deti...
  PESTEL (200 char): ## Analisis PESTEL (Regulasi & Makro Indonesia)

**Political & Economic:**
- Stabilitas politik Indonesia pasca-pemilu 2024 mendukung iklim investasi teknologi dengan insentif tax allowance untuk sekt...


### 7.3 Financial Analyst Agent

In [29]:
from nodes.financial_analyst import financial_analyst_node

state_fa = make_fresh_state(iteration=0, with_market=True, with_strategic=True)
# state_fa['market_scout_report'] = None  
state_fa["market_scout_report"] = report_ms
if report_sa is not None:
    state_fa["strategic_report"] = report_sa

print("Menjalankan Financial Analyst Agent...")
result_fa = financial_analyst_node(state_fa)

report_fa = result_fa.get("financial_analysis_report")
assert report_fa is not None, "financial_analysis_report seharusnya tidak None!"
assert report_fa.analysis_result, "analysis_result tidak boleh kosong"

print("\n[Financial Analyst] LULUS")
print(f"  Hasil (300 char) : {report_fa.analysis_result[:300]}...")

2026-05-27 12:27:00.127  DEBUG  [clario.financial_analyst_enterprise]  ============================================================
2026-05-27 12:27:00.128  DEBUG  [clario.financial_analyst_enterprise]  -> Enterprise Financial Analyst Agent Dimulai
2026-05-27 12:27:00.136  DEBUG  [clario.enterprise_financial_analyst]  [PLAN] Merencanakan 2 query pencarian...
[12:27:00] Request options: {'method': 'post', 'url': '/chat/completions', 'headers': {'X-Stainless-Raw-Response': 'true'}, 'files': None, 'idempotency_key': 'stainless-python-retry-2e1dfb2f-a688-4f01-8540-bc2961773c12', 'security': {'bearer_auth': True}, 'content': None, 'json_data': {'messages': [{'content': 'You are the Senior Financial Analyst Agent in an enterprise business planning system.\nYour mission is to act as a dynamic financial calculator and decision system that processes quantitative constraints into actionable viability reports.\n\nYou must parse the raw financial inputs (capital/modal, price/harga produk, traffic 

Menjalankan Financial Analyst Agent...


[12:27:00] start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x000001FF1BDAFD10>
[12:27:00] send_request_headers.started request=<Request [b'POST']>
[12:27:00] send_request_headers.complete
[12:27:00] send_request_body.started request=<Request [b'POST']>
[12:27:00] send_request_body.complete
[12:27:00] receive_response_headers.started request=<Request [b'POST']>
[12:27:42] receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Wed, 27 May 2026 05:27:42 GMT'), (b'Content-Type', b'application/json'), (b'Content-Length', b'17844'), (b'Connection', b'keep-alive'), (b'server', b'uvicorn'), (b'x-robots-tag', b'noindex'), (b'x-request-id', b'R6cGGQuovI5YDFiQsNdrUhn8')])
[12:27:42] HTTP Request: POST https://api.deepinfra.com/v1/openai/chat/completions "HTTP/1.1 200 OK"
[12:27:42] receive_response_body.started request=<Request [b'POST']>
[12:27:42] receive_response_body.complete
[12:27:42] response_closed.started
[12:27:42] response_cl


[Financial Analyst] LULUS
  Hasil (300 char) : # 📊 Laporan Analisis Kelayakan Finansial
## Platform Micro-Learning EdTech Indonesia

---

## 1. Ringkasan Eksekutif

Berdasarkan analisis kuantitatif terhadap model bisnis platform micro-learning untuk persiapan SNBT dan sertifikasi profesional, proyek ini menunjukkan **potensi kelayakan dengan cat...


In [30]:
print(f"  Hasil (300 char) : {report_fa.analysis_result}...")

  Hasil (300 char) : # 📊 Laporan Analisis Kelayakan Finansial
## Platform Micro-Learning EdTech Indonesia

---

## 1. Ringkasan Eksekutif

Berdasarkan analisis kuantitatif terhadap model bisnis platform micro-learning untuk persiapan SNBT dan sertifikasi profesional, proyek ini menunjukkan **potensi kelayakan dengan catatan kritis** pada fase awal operasional.

---

## 2. Asumsi Dasar Perhitungan

| Parameter | Nilai Asumsi |
|-----------|-------------|
| Harga Subscription | Rp 75.000/bulan |
| Target Subscriber Bulan 1-6 | 2.000 user |
| COGS (Server, Content, Payment Gateway) | 30% dari revenue |
| Biaya Pemasaran | 20% dari revenue |
| Modal Awal | Rp 500.000.000 |
| Gaji Tim (7 orang) | Rp 89.000.000/bulan |
| Sewa Kantor | Rp 15.000.000/bulan |
| Operasional Lain | Rp 10.000.000/bulan |

---

## 3. Proyeksi Arus Kas Bulanan (Kondisi Normal)

```
PENDAPATAN:
  Revenue Subscription (2.000 × Rp 75.000)    Rp 150.000.000

BIAYA VARIABEL (COGS):
  Server & Infrastructure              

In [31]:
report_ms.ideas

['Platform micro-learning video pendek (3-5 menit) khusus persiapan SNBT dengan fitur adaptive learning dan harga subscription terjangkau (Rp 50.000-100.000/bulan) untuk mengatasi keluhan harga mahal dan materi tidak update di kompetitor',
 'Layanan bundled micro-learning untuk sertifikasi profesional (TOEFL, AWS, Google Certifications) dengan sistem progress tracking real-time dan komunitas peer support, mengisi celah layanan paska-SMA yang belum optimal di pasar EdTech Indonesia']

In [32]:
print(report_fa.analysis_result)

# 📊 Laporan Analisis Kelayakan Finansial
## Platform Micro-Learning EdTech Indonesia

---

## 1. Ringkasan Eksekutif

Berdasarkan analisis kuantitatif terhadap model bisnis platform micro-learning untuk persiapan SNBT dan sertifikasi profesional, proyek ini menunjukkan **potensi kelayakan dengan catatan kritis** pada fase awal operasional.

---

## 2. Asumsi Dasar Perhitungan

| Parameter | Nilai Asumsi |
|-----------|-------------|
| Harga Subscription | Rp 75.000/bulan |
| Target Subscriber Bulan 1-6 | 2.000 user |
| COGS (Server, Content, Payment Gateway) | 30% dari revenue |
| Biaya Pemasaran | 20% dari revenue |
| Modal Awal | Rp 500.000.000 |
| Gaji Tim (7 orang) | Rp 89.000.000/bulan |
| Sewa Kantor | Rp 15.000.000/bulan |
| Operasional Lain | Rp 10.000.000/bulan |

---

## 3. Proyeksi Arus Kas Bulanan (Kondisi Normal)

```
PENDAPATAN:
  Revenue Subscription (2.000 × Rp 75.000)    Rp 150.000.000

BIAYA VARIABEL (COGS):
  Server & Infrastructure                      Rp 20.000.000

In [33]:
print(report_fa.analysis_result)

# 📊 Laporan Analisis Kelayakan Finansial
## Platform Micro-Learning EdTech Indonesia

---

## 1. Ringkasan Eksekutif

Berdasarkan analisis kuantitatif terhadap model bisnis platform micro-learning untuk persiapan SNBT dan sertifikasi profesional, proyek ini menunjukkan **potensi kelayakan dengan catatan kritis** pada fase awal operasional.

---

## 2. Asumsi Dasar Perhitungan

| Parameter | Nilai Asumsi |
|-----------|-------------|
| Harga Subscription | Rp 75.000/bulan |
| Target Subscriber Bulan 1-6 | 2.000 user |
| COGS (Server, Content, Payment Gateway) | 30% dari revenue |
| Biaya Pemasaran | 20% dari revenue |
| Modal Awal | Rp 500.000.000 |
| Gaji Tim (7 orang) | Rp 89.000.000/bulan |
| Sewa Kantor | Rp 15.000.000/bulan |
| Operasional Lain | Rp 10.000.000/bulan |

---

## 3. Proyeksi Arus Kas Bulanan (Kondisi Normal)

```
PENDAPATAN:
  Revenue Subscription (2.000 × Rp 75.000)    Rp 150.000.000

BIAYA VARIABEL (COGS):
  Server & Infrastructure                      Rp 20.000.000

### 7.4 Ethics Guardian Agent

In [34]:
from nodes.ethics_agent import ethics_agent_node

state_ea = make_fresh_state(iteration=0, with_market=True, with_strategic=True, with_financial=True)
state_ea["market_scout_report"] = report_ms
if report_sa: state_ea["strategic_report"] = report_sa
if report_fa: state_ea["financial_analysis_report"] = report_fa

print("Menjalankan Ethics Guardian Agent...")
result_ea = ethics_agent_node(state_ea)

report_ea = result_ea.get("ethics_analysis_report")
assert report_ea is not None, "ethics_analysis_report seharusnya tidak None!"
assert report_ea.analysis_result, "analysis_result tidak boleh kosong"

print("\n[Ethics Guardian] LULUS")
print(f"  Hasil (300 char) : {report_ea.analysis_result[:300]}...")

2026-05-27 12:30:55.807  DEBUG  [clario.ethics_agent_enterprise]  ============================================================
2026-05-27 12:30:55.808  DEBUG  [clario.ethics_agent_enterprise]  -> Enterprise Ethics & Compliance Agent Dimulai
2026-05-27 12:30:55.816  DEBUG  [clario.enterprise_ethics_guardian]  [PLAN] Merencanakan 2 query pencarian...
[12:30:55] Request options: {'method': 'post', 'url': '/chat/completions', 'headers': {'X-Stainless-Raw-Response': 'true'}, 'files': None, 'idempotency_key': 'stainless-python-retry-ae4af0c3-cafc-4ea8-923d-ed1306186db1', 'security': {'bearer_auth': True}, 'content': None, 'json_data': {'messages': [{'content': 'You are the Senior Ethics & Compliance Agent in an enterprise business planning system.\nYour mission is to act as a rules-based legal validation engine that identifies mandatory licensing, calculates regulatory risks, and builds operational compliance checklists under Indonesian law.\n\nYour analysis must be compiled entirely in Baha

Menjalankan Ethics Guardian Agent...


[12:30:56] start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x000001FF1C53ADE0>
[12:30:56] send_request_headers.started request=<Request [b'POST']>
[12:30:56] send_request_headers.complete
[12:30:56] send_request_body.started request=<Request [b'POST']>
[12:30:56] send_request_body.complete
[12:30:56] receive_response_headers.started request=<Request [b'POST']>
[12:31:32] receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Wed, 27 May 2026 05:31:32 GMT'), (b'Content-Type', b'application/json'), (b'Content-Length', b'16528'), (b'Connection', b'keep-alive'), (b'server', b'uvicorn'), (b'x-robots-tag', b'noindex'), (b'x-request-id', b'Raib9MXQf7JqIDugeHO4hJkM')])
[12:31:32] HTTP Request: POST https://api.deepinfra.com/v1/openai/chat/completions "HTTP/1.1 200 OK"
[12:31:32] receive_response_body.started request=<Request [b'POST']>
[12:31:32] receive_response_body.complete
[12:31:32] response_closed.started
[12:31:32] response_cl


[Ethics Guardian] LULUS
  Hasil (300 char) : # 📋 LAPORAN ANALISIS KEPATUHAN HUKUM
## Platform Micro-Learning EdTech Indonesia

---

## 1. Ringkasan Eksekutif

Berdasarkan analisis terhadap model bisnis platform micro-learning untuk persiapan SNBT dan sertifikasi profesional, proyek ini **dapat beroperasi secara legal** dengan memenuhi persyara...


### 7.5 Lead Orchestrator — iterasi pertama (belum ada laporan)

In [35]:
from nodes.lead_orchestrator import lead_orchestrator_node

# Iterasi 0: semua report None → orchestrator harus route ke market scout
state_orch_init = make_fresh_state(iteration=0)
print("Menjalankan Lead Orchestrator (iterasi awal, tanpa laporan)...")
result_orch_init = lead_orchestrator_node(state_orch_init)

print("\n[Orchestrator iterasi awal] Output:")
print(f"  approval_status      : {result_orch_init.get('approval_status')}")
print(f"  orchestrator_feedback: {str(result_orch_init.get('orchestrator_feedback', ''))[:150]}")
print(f"  iteration            : {result_orch_init.get('iteration')}")

2026-05-27 12:32:38.352  DEBUG  [clario.lead_orchestrator]  ============================================================


2026-05-27 12:32:38.353  DEBUG  [clario.lead_orchestrator]  → Lead Orchestrator dimulai
2026-05-27 12:32:38.354  DEBUG  [clario.lead_orchestrator]    First pass — belum ada report, routing ke Market Scout
2026-05-27 12:32:38.355  DEBUG  [clario.lead_orchestrator]  ✓ Lead Orchestrator selesai dalam 0.00s
2026-05-27 12:32:38.356  DEBUG  [clario.lead_orchestrator]  ============================================================


Menjalankan Lead Orchestrator (iterasi awal, tanpa laporan)...

[Orchestrator iterasi awal] Output:
  approval_status      : pending
  orchestrator_feedback: None
  iteration            : None


In [36]:
# Iterasi 1: semua report sudah ada → orchestrator evaluasi dan beri verdict
state_orch_eval = make_fresh_state(iteration=1, with_market=True, with_strategic=True, with_financial=True, with_ethics=True)
state_orch_eval["market_scout_report"] = report_ms
if report_sa: state_orch_eval["strategic_report"] = report_sa
if report_fa: state_orch_eval["financial_analysis_report"] = report_fa
if report_ea: state_orch_eval["ethics_analysis_report"] = report_ea

print("Menjalankan Lead Orchestrator (evaluasi laporan lengkap)...")
result_orch_eval = lead_orchestrator_node(state_orch_eval)

status = result_orch_eval.get("approval_status")
assert status in ("approved", "rejected", "pending"), f"Status tidak valid: {status}"

print("\n[Orchestrator evaluasi] LULUS")
print(f"  approval_status      : {status}")
print(f"  iteration            : {result_orch_eval.get('iteration')}")
feedback = result_orch_eval.get('orchestrator_feedback') or ""
print(f"  feedback (200 char)  : {feedback[:200]}")

2026-05-27 12:36:29.900  DEBUG  [clario.lead_orchestrator]  ============================================================
2026-05-27 12:36:29.900  DEBUG  [clario.lead_orchestrator]  → Lead Orchestrator dimulai
2026-05-27 12:36:29.906  DEBUG  [clario.lead_orchestrator]  [LLM] Evaluasi iterasi 1 — memanggil LLM (Tree of Thoughts)...
[12:36:29] Request options: {'method': 'post', 'url': '/chat/completions', 'headers': {'X-Stainless-Raw-Response': 'true'}, 'files': None, 'idempotency_key': 'stainless-python-retry-e16ac2d5-9154-4a0c-ab23-22f7371322d9', 'security': {'bearer_auth': True}, 'content': None, 'json_data': {'messages': [{'content': 'You are the Lead Orchestrator of a multi-agent AI business planning system.\nYour role is to evaluate the quality of the business plan produced by four specialist agents, detect discrepancies, score confidence, and decide the next state.\n\nYou apply Tree of Thoughts (ToT) reasoning: you must explicitly consider MULTIPLE evaluation\nperspectives (market

Menjalankan Lead Orchestrator (evaluasi laporan lengkap)...


[12:36:30] connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x000001FF19FB0A10>
[12:36:30] start_tls.started ssl_context=<ssl.SSLContext object at 0x000001FF19741850> server_hostname='api.deepinfra.com' timeout=None
[12:36:30] start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x000001FF1B607CB0>
[12:36:30] send_request_headers.started request=<Request [b'POST']>
[12:36:30] send_request_headers.complete
[12:36:30] send_request_body.started request=<Request [b'POST']>
[12:36:30] send_request_body.complete
[12:36:30] receive_response_headers.started request=<Request [b'POST']>
[12:37:06] receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Wed, 27 May 2026 05:37:07 GMT'), (b'Content-Type', b'application/json'), (b'Content-Length', b'9751'), (b'Connection', b'keep-alive'), (b'server', b'uvicorn'), (b'x-robots-tag', b'noindex'), (b'x-request-id', b'RvS7utJ02lSbPEAF0jCKJ47I')])
[12:37:06] HTTP Request

RuntimeError: Called get_config outside of a runnable context

---
## 8. Graph Pipeline Test

Uji build graph LangGraph dan jalankan **satu iterasi penuh** (satu putaran pipeline semua agent).

In [37]:
from graphs.ebp_graph import build_graph, _route_from_orchestrator
print("Import graphs.ebp_graph: OK")

Import graphs.ebp_graph: OK


In [38]:
# --- Test routing function secara unit ---

# Case 1: approved → END
route_approved = _route_from_orchestrator({"approval_status": "approved", "iteration": 1, "max_iterations": 3})
assert route_approved == "__end__", f"Expected __end__, got {route_approved}"
print("Route approved     :", route_approved)

# Case 2: max_iterations tercapai → END
route_maxed = _route_from_orchestrator({"approval_status": "rejected", "iteration": 3, "max_iterations": 3})
assert route_maxed == "__end__"
print("Route max_iter hit :", route_maxed)

# Case 3: rejected, masih ada iterasi → kembali ke market_scout
route_retry = _route_from_orchestrator({"approval_status": "rejected", "iteration": 1, "max_iterations": 3})
assert route_retry == "market_scout"
print("Route retry        :", route_retry)

print("\nSemua routing case: LULUS")

AssertionError: Expected __end__, got final_summary

In [39]:
# --- Build graph ---
graph = build_graph()
print("Graph berhasil dikompilasi:", type(graph).__name__)
print("Nodes:", list(graph.nodes.keys()) if hasattr(graph, 'nodes') else "(gunakan graph.get_graph() untuk detail)")

Graph berhasil dikompilasi: CompiledStateGraph
Nodes: ['__start__', 'lead_orchestrator', 'market_scout', 'strategic_architect', 'financial_analyst', 'ethics_agent', 'final_summary']


---
## 9. End-to-End Integration Test

Jalankan graph secara penuh dengan **max_iterations=1** agar hanya 1 siklus pipeline yang berjalan. Ini mensimulasikan alur produksi nyata.

> **Perhatian:** Sel ini memanggil semua 5 agent dan internet search. Estimasi waktu: **5-15 menit**.

In [41]:
import time

e2e_constraints = BussinessConstraints(
    sector_and_domain="EdTech platform untuk pembelajaran data science online",
    audience="Pelajar SMA dan mahasiswa di Indonesia (usia 16-22)",
    initial_prompt="Platform micro-learning berbasis video pendek dengan compiler untuk membangun end to end data science projects online",
)

initial_state: EBPState = {
    "state_id": str(uuid.uuid4()),
    "user_id": "e2e_test",
    "bussiness_constraints": e2e_constraints,
    "market_scout_report": None,
    "strategic_report": None,
    "financial_analysis_report": None,
    "ethics_analysis_report": None,
    "approval_status": "pending",
    "orchestrator_feedback": None,
    "messages": [],
    "iteration": 0,
    "max_iterations": 1,  # 1 iterasi saja untuk test cepat
    "user_feedback": None,
}

print(f"State ID: {initial_state['state_id']}")
print("Menjalankan graph end-to-end (max_iterations=1)...")
print("=" * 60)

config = {"configurable": {"thread_id": "uji_coba"}}
start = time.time()
final_state = graph.invoke(initial_state, config=config)
elapsed = time.time() - start

print(f"\nGraph selesai dalam {elapsed:.1f} detik")

2026-05-27 12:44:22.726  DEBUG  [clario.lead_orchestrator]  ============================================================
2026-05-27 12:44:22.727  DEBUG  [clario.lead_orchestrator]  → Lead Orchestrator dimulai
2026-05-27 12:44:22.728  DEBUG  [clario.lead_orchestrator]    First pass — belum ada report, routing ke Market Scout
2026-05-27 12:44:22.728  DEBUG  [clario.lead_orchestrator]  ✓ Lead Orchestrator selesai dalam 0.00s
2026-05-27 12:44:22.729  DEBUG  [clario.lead_orchestrator]  ============================================================
2026-05-27 12:44:22.733  DEBUG  [clario.market_scout_enterprise]  ============================================================
2026-05-27 12:44:22.734  DEBUG  [clario.market_scout_enterprise]  -> Enterprise Market Scout Agent Dimulai
2026-05-27 12:44:22.741  DEBUG  [clario.enterprise_market_scout]  [PLAN] Merencanakan 4 query pencarian...
[12:44:22] Request options: {'method': 'post', 'url': '/chat/completions', 'headers': {'X-Stainless-Raw-Response

State ID: f2559a94-7eef-4434-8120-d41970637094
Menjalankan graph end-to-end (max_iterations=1)...


[12:44:23] connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x000001FF1C5B3200>
[12:44:23] start_tls.started ssl_context=<ssl.SSLContext object at 0x000001FF19741850> server_hostname='api.deepinfra.com' timeout=None
[12:44:23] start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x000001FF1C248860>
[12:44:23] send_request_headers.started request=<Request [b'POST']>
[12:44:23] send_request_headers.complete
[12:44:23] send_request_body.started request=<Request [b'POST']>
[12:44:23] send_request_body.complete
[12:44:23] receive_response_headers.started request=<Request [b'POST']>
[12:44:50] receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Wed, 27 May 2026 05:44:51 GMT'), (b'Content-Type', b'application/json'), (b'Content-Length', b'7109'), (b'Connection', b'keep-alive'), (b'server', b'uvicorn'), (b'x-robots-tag', b'noindex'), (b'x-request-id', b'Rf0m6QiZevGSGA2whLcTzVuv')])
[12:44:50] HTTP Request


Graph selesai dalam 414.4 detik


In [42]:
# --- Validasi output end-to-end ---
print("=" * 60)
print("HASIL END-TO-END")
print("=" * 60)

print(f"approval_status     : {final_state.get('approval_status')}")
print(f"iteration           : {final_state.get('iteration')}")
print(f"messages count      : {len(final_state.get('messages', []))}")

ms = final_state.get("market_scout_report")
sa = final_state.get("strategic_report")
fa = final_state.get("financial_analysis_report")
ea = final_state.get("ethics_analysis_report")

print(f"\nmarket_scout_report      : {'Ada' if ms else 'TIDAK ADA'}")
print(f"strategic_report         : {'Ada' if sa else 'TIDAK ADA'}")
print(f"financial_analysis_report: {'Ada' if fa else 'TIDAK ADA'}")
print(f"ethics_analysis_report   : {'Ada' if ea else 'TIDAK ADA'}")

HASIL END-TO-END
approval_status     : rejected
iteration           : 1
messages count      : 27

market_scout_report      : Ada
strategic_report         : Ada
financial_analysis_report: Ada
ethics_analysis_report   : Ada


In [44]:
# --- Tampilkan ringkasan tiap laporan ---
DIVIDER = "-" * 60

if ms:
    print(DIVIDER)
    print("MARKET SCOUT")
    print(DIVIDER)
    for i, idea in enumerate(ms.ideas, 1):
        print(f"  Ide {i}: {idea}")
    print(f"\n  Penjelasan: {ms.agent_explanation[:400]}")

if sa:
    print(DIVIDER)
    print("STRATEGIC ARCHITECT")
    print(DIVIDER)
    print(f"  SWOT  : {sa.swot_analysis[:400]}")
    print(f"  PESTEL: {sa.pastel_analysis[:400]}")

if fa:
    print(DIVIDER)
    print("FINANCIAL ANALYST")
    print(DIVIDER)
    print(f"  {fa.analysis_result[:1500]}")

if ea:
    print(DIVIDER)
    print("ETHICS GUARDIAN")
    print(DIVIDER)
    print(f"  {ea.analysis_result[:500]}")

feedback = final_state.get("orchestrator_feedback") or ""
if feedback:
    print(DIVIDER)
    print("ORCHESTRATOR FEEDBACK")
    print(DIVIDER)
    print(f"  {feedback[:400]}")

------------------------------------------------------------
MARKET SCOUT
------------------------------------------------------------
  Ide 1: Platform micro-learning data science dengan video pendek (5-10 menit) dilengkapi online compiler Python/R untuk project end-to-end, menjawab keluhan materi terlalu panjang dan tidak ada praktik langsung di kompetitor seperti Dicoding dan RevoU
  Ide 2: Model subscription terjangkau (Rp 50.000-150.000/bulan) dengan akses unlimited ke semua modul data science, solusi terhadap keluhan harga bootcamp yang terlalu mahal (Rp 5-15 juta) di Pacmann AI dan RevoU

  Penjelasan: Berdasarkan analisis multi-source, pasar EdTech data science untuk pelajar dan mahasiswa Indonesia menunjukkan kelayakan komersial yang kuat. Total Addressable Market (TAM) mencapai Rp 17,25 triliun dengan basis 23 juta target audience usia 16-22 tahun dan estimasi spending Rp 750.000/tahun. Serviceable Addressable Market (SAM) sebesar Rp 6,9 triliun memperhitungkan penetrasi inte

In [45]:
# --- Assertions akhir end-to-end ---
assert final_state.get("approval_status") in ("approved", "rejected", "pending"), \
    "approval_status harus valid"
assert final_state.get("market_scout_report") is not None, \
    "Market scout report seharusnya terisi"
assert final_state.get("strategic_report") is not None, \
    "Strategic report seharusnya terisi"
assert final_state.get("financial_analysis_report") is not None, \
    "Financial report seharusnya terisi"
assert final_state.get("ethics_analysis_report") is not None, \
    "Ethics report seharusnya terisi"

print("SEMUA ASSERTION END-TO-END: LULUS")
print(f"\nSistem ClarioAI berjalan normal. Status akhir: {final_state['approval_status'].upper()}")

SEMUA ASSERTION END-TO-END: LULUS

Sistem ClarioAI berjalan normal. Status akhir: REJECTED
